In [ ]:
from google.colab import drive
drive.mount('/content/drive/')

# CIFAR-10 — MLP Classification

We compare two architecturally distinct MLPs on CIFAR-10:
- **Architecture 1 — Logistic Regression**: linear baseline (no hidden layers)
- **Architecture 2 — Shallow MLP**: 1 hidden layer, 512 units, ReLU
- **Architecture 3 — Deep MLP**: 3 hidden layers (1024 → 512 → 256), BatchNorm, Dropout, Early Stopping

> CIFAR-10 is much harder for MLPs than Fashion-MNIST because colour images
> have strong spatial structure that flat vectors cannot exploit.

### Imports

In [ ]:
import gc
import matplotlib.pyplot as plt
import numpy as np
import tensorflow as tf
%matplotlib inline

from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
from tensorflow.keras import backend as K
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras.datasets import cifar10
from tensorflow.keras.layers import Input, Dense, Dropout, BatchNormalization
from tensorflow.keras.models import Model
from tensorflow.keras.optimizers import SGD, Adam
from tensorflow.keras.utils import to_categorical
from tensorflow.random import set_seed

print("Tensorflow version " + tf.__version__)

### Data loading & preprocessing

In [ ]:
batch_size = 128
classes    = 10
epochs     = 100
input_dim  = 32 * 32 * 3  # 3072

class_names = ['airplane','automobile','bird','cat','deer',
               'dog','frog','horse','ship','truck']

(X_train, y_train), (X_test, y_test) = cifar10.load_data()

X_train = X_train.reshape(X_train.shape[0], input_dim).astype('float32') / 255
X_test  = X_test.reshape(X_test.shape[0],   input_dim).astype('float32') / 255

Y_train = to_categorical(y_train, classes)
Y_test  = to_categorical(y_test,  classes)

print(f'Train: {X_train.shape}, Test: {X_test.shape}')

### Visualise samples

In [ ]:
plt.style.use('dark_background')
fig, axes = plt.subplots(2, 5, figsize=(12, 5))
for i, ax in enumerate(axes.flat):
    ax.imshow(X_train[i].reshape(32, 32, 3))
    ax.set_title(class_names[y_train[i][0]], fontsize=9)
    ax.axis('off')
plt.suptitle('CIFAR-10 — sample images', fontsize=13)
plt.tight_layout()
plt.show()

### Helper functions

In [ ]:
def plot_history(hs, epochs, metric):
    """hs: dict of {label: history_object}"""
    plt.style.use('dark_background')
    plt.rcParams['figure.figsize'] = [15, 8]
    plt.rcParams['font.size'] = 16
    plt.clf()
    for label, hist in hs.items():
        plt.plot(hist.history[metric],
                 label=f'{label} train {metric}', linewidth=2)
        plt.plot(hist.history[f'val_{metric}'],
                 label=f'{label} val {metric}', linewidth=2)
    x_ticks = np.arange(0, epochs + 1, max(1, epochs // 10))
    x_ticks[0] += 1
    plt.xticks(x_ticks)
    plt.ylim((0, 1))
    plt.xlabel('Epochs')
    plt.ylabel('Loss' if metric == 'loss' else 'Accuracy')
    plt.legend()
    plt.show()


def plot_confusion_matrix(model, title='Confusion Matrix'):
    y_pred = np.argmax(model.predict(X_test, verbose=0), axis=1)
    cm = confusion_matrix(y_test, y_pred)
    plt.style.use('dark_background')
    fig, ax = plt.subplots(figsize=(10, 8))
    disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=class_names)
    disp.plot(ax=ax, colorbar=True, xticks_rotation=45)
    ax.set_title(title, fontsize=13)
    plt.tight_layout()
    plt.show()


def clean_up(model):
    K.clear_session()
    del model
    gc.collect()

### Model builder

In [ ]:
def build_and_train(
        optimizer,
        hidden_units=None,
        dropout_rate=0.0,
        use_batchnorm=False,
        epochs=100,
        batch_size=128,
        callbacks=None,
        verbose=0):
    """
    hidden_units: list of ints. [] = Logistic Regression.
    """
    if hidden_units is None:
        hidden_units = []

    np.random.seed(1402)
    set_seed(1981)

    inp = Input(shape=(input_dim,), name='Input')
    x = inp
    for i, units in enumerate(hidden_units):
        x = Dense(units, activation='relu',
                  kernel_initializer='glorot_uniform',
                  name=f'Hidden-{i+1}')(x)
        if use_batchnorm:
            x = BatchNormalization(name=f'BN-{i+1}')(x)
        if dropout_rate > 0:
            x = Dropout(rate=dropout_rate, name=f'Dropout-{i+1}')(x)

    out = Dense(classes, activation='softmax',
                kernel_initializer='glorot_uniform', name='Output')(x)

    model = Model(inputs=inp, outputs=out)
    model.compile(optimizer=optimizer,
                  loss='categorical_crossentropy',
                  metrics=['accuracy'])
    hs = model.fit(
        X_train, Y_train,
        validation_split=0.1,
        epochs=epochs,
        batch_size=batch_size,
        callbacks=callbacks,
        verbose=verbose
    )
    print('Finished training.')
    print('------------------')
    model.summary()
    return model, hs

---
## Architecture 1 — Logistic Regression (linear baseline)
Input (3072) → Output (10, softmax). No hidden layers.

In [ ]:
lr_model, lr_hs = build_and_train(
    optimizer=Adam(), hidden_units=[], epochs=epochs, batch_size=batch_size
)
lr_eval = lr_model.evaluate(X_test, Y_test, verbose=1)

print(f"\nTest Acc: {lr_eval[1]:.5f}  Test Loss: {lr_eval[0]:.5f}")
plot_history({'LR': lr_hs}, epochs, 'loss')
plot_history({'LR': lr_hs}, epochs, 'accuracy')

clean_up(lr_model)

---
## Architecture 2 — Shallow MLP
Input (3072) → Dense(512, ReLU) → Output (10, softmax)

In [ ]:
shallow_model, shallow_hs = build_and_train(
    optimizer=Adam(), hidden_units=[512], epochs=epochs, batch_size=batch_size
)
shallow_eval = shallow_model.evaluate(X_test, Y_test, verbose=1)

print(f"\nTest Acc: {shallow_eval[1]:.5f}  Test Loss: {shallow_eval[0]:.5f}")
plot_history({'LR': lr_hs, 'Shallow MLP [512]': shallow_hs}, epochs, 'loss')
plot_history({'LR': lr_hs, 'Shallow MLP [512]': shallow_hs}, epochs, 'accuracy')

clean_up(shallow_model)

---
## Architecture 3 — Deep MLP
Input (3072) → Dense(1024) → BN → Drop(0.3) → Dense(512) → BN → Drop(0.3) → Dense(256) → BN → Drop(0.3) → Output(10)

In [ ]:
es = EarlyStopping(monitor='val_accuracy', patience=15,
                   verbose=1, restore_best_weights=True)

deep_model, deep_hs = build_and_train(
    optimizer=Adam(),
    hidden_units=[1024, 512, 256],
    dropout_rate=0.3,
    use_batchnorm=True,
    epochs=epochs,
    batch_size=batch_size,
    callbacks=[es],
    verbose=1
)
deep_eval = deep_model.evaluate(X_test, Y_test, verbose=1)

ep = len(deep_hs.history['loss'])
print(f"\nStopped @ epoch {ep}")
print(f"Test Acc: {deep_eval[1]:.5f}  Test Loss: {deep_eval[0]:.5f}")
plot_history({'Deep MLP [1024,512,256]': deep_hs}, ep, 'loss')
plot_history({'Deep MLP [1024,512,256]': deep_hs}, ep, 'accuracy')

### Confusion matrix — best model (Deep MLP)

In [ ]:
plot_confusion_matrix(deep_model, title='Confusion Matrix — Deep MLP (CIFAR-10)')
clean_up(deep_model)

---
## Final comparison — all MLP models on CIFAR-10

In [ ]:
print(f"{'Model':<42} {'Test Acc':>10} {'Test Loss':>12}")
print('-' * 66)
rows = [
    ('Logistic Regression',              lr_eval),
    ('Shallow MLP [512]',                shallow_eval),
    ('Deep MLP [1024,512,256]+BN+Drop',  deep_eval),
]
for name, ev in rows:
    print(f"{name:<42} {ev[1]:>10.5f} {ev[0]:>12.5f}")